### 🎯 허깅페이스에서 위스퍼 모델 내려받아 로컬에서 사용해보기
#### 📚 필요한 패키지 설치
##### 📗 Library

In [ ]:
%uv add --upgrade transformers "datasets[audio]" accelerate

- transformers: Hugging Face Transformers 라이브러리
- datasets: Hugging Face 데이터셋 라이브러리
- datasets[audio]: datasets의 audio 추가 기능까지 설치
- accelerate: 여러 GPU, CPU, 분산 실행 등을 돕는 라이브러리
- --upgrade: 기존에 설치된 패키지가 있으면 가능한 최신 버전으로 업데이트

##### 📕FFMPEG
- 오디오와 비디오 파일을 변환해 처리하는 오픈소스 도구.
- 위스퍼 로컬모델 사용을 위해 필요하다.
- https://www.gyan.dev/ffmpeg/builds/

In [1]:
import os
os.environ["PATH"] += os.pathsep + r"D:\ksol927_Study_Ai\ffmpeg-2026-08-20-git-7d77562d2a-full_build\bin"

##### 📘PyTorch
- 페이스북의 AI 리서치 랩 FAIR에서 개발한 오픈소스 딥러닝 라이브러리.
- https://pytorch.org

In [ ]:
%pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132

---
#### 📖 허깅페이스에서 제공한 코드 활용하기

In [7]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# 샘플 데이터 대신 로컬 MP3 파일 사용
# from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,   # 청크별로 타임스탬프 반환
    chunk_length_s=10,  # 입력 오디오 10초씩 나누기
    stride_length_s=2,  # 2초씩 겹치도록 청크 나누기
) 

# 샘플 데이터 대신 로컬 MP3 파일 사용
# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
sample = "audio/lsy_audio_2023_58s.mp3"

result = pipe(sample)

c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ksol9\.cache\huggingface\hub\models--openai--whisper-large-v3-turbo. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Deve

##### ✏️ 결과값
- text : MP3 파일의 모든 내용
- chunks : 시간대별로 어떤 내용이 들어있는지 정리

In [8]:
from rich.pretty import pprint
pprint(result)

{
│   'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT-API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고',
│   'chunks': [
│   │   {
│   │   │   'timestamp': (0.0, 6.3),
│   │   │   'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'
│   │   },
│   │   {'timestamp': (7.18, 10.0), 'text': ' GPT-API에 대해서 생소하신 분들도 있을텐데'},
│   │   {'timestamp': (11.0, 17.0), 'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'},
│   │   {'timestamp': (17.0, 20.0), 'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'},
│   │   {'timestamp': (20.0, 22.0), 'text': ' 이야기할 거예요.'},
│   │   {'timestamp': (22.0, 24.0), 'text': ' 그래서 이런 강의들이 사실 많이 있습니다.'},
│   │   {'timestamp': (24.0, 27.48), 'text': ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'},
│   │   {'timestamp': (27.48, 29.58), 'text': ' GPT로 명확한 미션을 달성하는'},
│   │   {'timestamp': (29.58, 31.66), 'text': ' 챕터 프로그램을 만드는게 사실'},
│   │   {'timestamp': (31.66, 34.32), 'text': ' 쉽지는 않은데 이걸 어떻게 해서'},
│   │   {'timestamp': (34.32, 36.4), 'text': ' 구현을 하는지 그리고 그게 왜 필요한지에 대해서'},
│   │   {'timestamp': (36.4, 37.34), 'text': ' 좀 이야기를 할 거고요.'},
│   │   {'timestamp': (38.0, 40.0), 'text': ' 그 예제로'},
│   │   {'timestamp': (40.0, 42.0), 'text': ' 예제는 여러가지가 될 수 있는데'},
│   │   {
│   │   │   'timestamp': (42.0, 45.66),
│   │   │   'text': ' 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서'
│   │   },
│   │   {'timestamp': (45.66, 47.1), 'text': ' 생성하는 프로그램을 만드는 것을'},
│   │   {'timestamp': (47.1, 48.46), 'text': ' 다루려고 합니다.'},
│   │   {'timestamp': (49.84, 51.96), 'text': ' 프로그램이 실행되는 모습을 한번 보여드릴게요.'},
│   │   {'timestamp': (52.84, 58.0), 'text': ' 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고'}
│   ]
}

##### ✏️ chunks 값을 CSV 파일로 저장
- 판다스 데이터프레임을 이용해 저장.
- 데이터프레임이란 데이터를 행과 열로 구성한 2차원 표 형태로 정리한 데이터 구조.

In [9]:
start_end_text = []

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index=False, sep="|")
display(df)

,start,end,text
0,0.00,6.30,안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.
1,7.18,10.00,GPT-API에 대해서 생소하신 분들도 있을텐데
2,11.00,17.00,"우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서"
3,17.00,20.00,우리가 원하는 프로그램을 어떻게 만드는지에 대해서
4,20.00,22.00,이야기할 거예요.
5,22.00,24.00,그래서 이런 강의들이 사실 많이 있습니다.
6,24.00,27.48,그래서 여러가지들이 있는데 이 강의 특징이라고 한다면
7,27.48,29.58,GPT로 명확한 미션을 달성하는
8,29.58,31.66,챕터 프로그램을 만드는게 사실
9,31.66,34.32,쉽지는 않은데 이걸 어떻게 해서
